In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
from matplotlib import pyplot as plt
import pycountry_convert as pc
import pandas as pd
import seaborn as sns
import arviz as az
import pycountry

from emu_renewal.inputs import OUTPUTS_PATH, get_ordered_countries_by_cont
from emu_renewal.plotting import G_MOB_DOMAIN_CMAP, A_MOB_DOMAIN_CMAP, get_cont_mobility
from emu_renewal.utils import get_countries_by_continent

In [ ]:
n_countries_per_cont = 5
mob_type = "g_mob"

In [ ]:
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
conts = [cont for cont in countries_by_cont if cont != "OC"]
ordered_countries = get_ordered_countries_by_cont(countries_by_cont, conts)

In [ ]:
fig, axes = plt.subplots(len(conts), n_countries_per_cont, figsize=[15, 15])
palette = G_MOB_DOMAIN_CMAP if mob_type == "g_mob" else A_MOB_DOMAIN_CMAP
for ct, cont in enumerate(conts):
    continent = pc.convert_continent_code_to_continent_name(cont)
    mob = get_cont_mobility(cont, countries_by_cont, mob_type)
    countries = [c for c in ordered_countries[cont] if c in mob][:n_countries_per_cont]
    for c, iso3 in enumerate(countries):
        c_path = job_path / iso3
        idata = az.from_netcdf(c_path / f"{mob_type}/idata_filtered.nc")
        weights = idata.posterior["mob_weights"].to_dataframe().unstack("mob_weights_dim_0")
        weights.columns = mob[iso3].columns
        ax = axes[ct, c]
        sns.kdeplot(weights, fill=True, alpha=0.05, linewidth=2.0, ax=ax, palette=palette)
        ax.set_title(pycountry.countries.lookup(iso3).name)
        ax.set_yticks([])
        if ct != len(conts) - 1 or c != 0:
            ax.get_legend().remove()
        if c == 0:
            ax.set_ylabel(continent, fontsize=20)
        else:
            ax.set_ylabel("")
    for a in range(c + 1, axes.shape[1]):
        ax = axes[ct, a]
        ax.set_axis_off()
fig.tight_layout()

In [ ]:
fig.savefig(f"{mob_type}_weights.pdf")